In [1]:
import pandas as pd


df = pd.read_csv('../data/online_retail.csv', encoding='unicode_escape')


df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.contains('C')]
df = df[df['Quantity'] > 0]


df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

display(df.head())


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom,20.34


In [2]:
import datetime as dt

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


today_date = df['InvoiceDate'].max() + dt.timedelta(days=1)


rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda date: (today_date - date.max()).days, # Recency (Son alışverişten geçen gün)
    'InvoiceNo': lambda num: num.nunique(),                     # Frequency (Farklı fatura sayısı)
    'TotalPrice': lambda price: price.sum()                     # Monetary (Bıraktığı toplam para)
})


rfm.columns = ['Recency', 'Frequency', 'Monetary']


rfm = rfm[rfm['Monetary'] > 0]

display(rfm.head())

C:\Users\kutuk\AppData\Local\Temp\ipykernel_20368\128590273.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


rfm_log = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']])


scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)


kmeans = KMeans(n_clusters=4, random_state=42)


rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)


rfm['Cluster'] = rfm['Cluster'].astype(str)

display(rfm.head())

,Recency,Frequency,Monetary,Cluster
CustomerID,,,,
12346.0,326,1,77183.60,3
12347.0,2,7,4310.00,1
12348.0,75,4,1797.24,3
12349.0,19,1,1757.55,0
12350.0,310,1,334.40,2


In [ ]:
import plotly.express as px

# 1. Kümeleri İngilizce sektör terimleriyle isimlendirmek için bir harita oluşturuyoruz
segment_map = {
        '1': 'Champions',
        '2': 'Lost Customers',
        '0': 'Potential Customers',
        '3': 'At Risk VIPs'
}

# 2. 'Cluster' sütunundaki rakamları eşleştirip yeni bir 'Segment' sütunu yaratıyoruz
rfm['Segment'] = rfm['Cluster'].map(segment_map)

# 3. Grafiği yeni 'Segment' sütununa göre çizdiriyoruz
fig = px.scatter_3d(
    rfm,
    x='Recency',
    y='Frequency',
    z='Monetary',
    color='Segment',          # Artık 0, 1, 2, 3 yerine isimlendirmeler görünecek
    opacity=0.7,
    title='3D RFM Customer Segmentation'
)

fig.update_layout(template='plotly_white')

fig.show()